# 07 — SHAP Explainability
## MediSight AI: A Transparency Label for Healthcare Prediction Systems

---

### What this notebook does

This notebook loads the already-trained models and generates SHAP
explanations for all 4 diseases.

It does NOT retrain any model. It does NOT reprocess any dataset.
It only reads what already exists and explains it.

### Output files (saved to ../explainability/)

| File | Purpose |
|---|---|
| `{disease}_shap_values.pkl` | Raw SHAP values for dashboard use |
| `{disease}_shap.json` | Feature importance (for dashboard JSON reads) |
| `{disease}_shap_bar.png` | Bar chart — global feature importance |
| `{disease}_shap_beeswarm.png` | Beeswarm — per-sample feature impact |
| `{disease}_shap_waterfall.png` | Single patient explanation |
| `{disease}_explainer.pkl` | Saved explainer for live predictions |

In [10]:
import sys
print("Python:", sys.version)
print("Path:", sys.executable)

Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
Path: c:\Users\neman\AppData\Local\Programs\Python\Python311\python.exe


In [11]:
import shap
import numpy as np
import pandas as pd
import joblib
import json
import os
import warnings
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
matplotlib.use("Agg")  # save plots to file, no pop-up windows

os.makedirs("../explainability", exist_ok=True)

print(f"SHAP version  : {shap.__version__}")
print(f"Output folder : ../explainability/  ")

SHAP version  : 0.51.0
Output folder : ../explainability/  


---

## Step 1 — Configuration

We have 3 trained models per disease (LR, RF, XGB).
We pick the best one for SHAP based on AUC-ROC.

| Disease | Model chosen | Reason |
|---|---|---|
| Diabetes | XGBoost (xgb) | Highest AUC 0.828 |
| Heart | Logistic Regression (lr) | Best recall 78% |
| Liver | Random Forest (rf) | Highest AUC 0.937 |
| Kidney | Logistic Regression (lr) | All 100% — LR is simplest |

### SHAP Explainer types

- **TreeExplainer** → XGBoost, Random Forest (fast, exact)
- **KernelExplainer** → Logistic Regression (model-agnostic, slower)

We use KernelExplainer for LR because LinearExplainer requires
unwrapping CalibratedClassifierCV — KernelExplainer works directly
on any model's predict_proba without unwrapping.

In [12]:
DISEASE_CONFIG = {
    "diabetes": {
        "model_key"   : "xgb",
        "explainer"   : "tree",
        "display_name": "Diabetes",
    },
    "heart": {
        "model_key"   : "lr",
        "explainer"   : "kernel",
        "display_name": "Heart Disease",
    },
    "liver": {
        "model_key"   : "rf",
        "explainer"   : "tree",
        "display_name": "Liver Disease",
    },
    "kidney": {
        "model_key"   : "lr",
        "explainer"   : "kernel",
        "display_name": "Chronic Kidney Disease",
    },
}

print("Configuration loaded:")
for disease, cfg in DISEASE_CONFIG.items():
    print(f"  {disease:<10} → model: {cfg['model_key'].upper():<5} "
          f"explainer: {cfg['explainer']}")

Configuration loaded:
  diabetes   → model: XGB   explainer: tree
  heart      → model: LR    explainer: kernel
  liver      → model: RF    explainer: tree
  kidney     → model: LR    explainer: kernel


---

## Step 2 — Core Functions

Four simple functions, each doing one job:

| Function | What it does |
|---|---|
| `load_data()` | Loads model + scaler + features + test data |
| `make_explainer()` | Creates the right SHAP explainer |
| `get_shap_values()` | Computes SHAP values, returns clean 2D array |
| `save_outputs()` | Saves all plots + JSON + pkl files |

In [13]:
def load_data(disease):
    """
    Loads everything needed for one disease.
    Returns: model, features (list), X_test (array), y_test (array)
    """
    cfg       = DISEASE_CONFIG[disease]
    model_key = cfg["model_key"]

    model    = joblib.load(f"../models/{disease}_{model_key}.pkl")
    scaler   = joblib.load(f"../models/{disease}_scaler.pkl")
    features = joblib.load(f"../datasets/processed/{disease}_features.pkl")
    X_test   = np.load(f"../datasets/processed/{disease}_X_test.npy")
    y_test   = np.load(f"../datasets/processed/{disease}_y_test.npy")

    print(f"  Model    : {type(model).__name__}")
    print(f"  Features : {len(features)}")
    print(f"  X_test   : {X_test.shape}")

    return model, features, X_test, y_test


print("load_data() defined ")

load_data() defined 


In [19]:
def make_explainer(model, explainer_type, X_test):
    """
    Creates the correct SHAP explainer.
    Automatically unwraps CalibratedClassifierCV if present.
    """
    # ── Unwrap CalibratedClassifierCV if needed ──────────────────────
    # All our models were wrapped during training for better probability
    # estimates. TreeExplainer needs the raw model inside the wrapper.
    if hasattr(model, "calibrated_classifiers_"):
        base_model = model.calibrated_classifiers_[0].estimator
        print(f"  Unwrapped CalibratedClassifierCV → {type(base_model).__name__}")
    else:
        base_model = model
        print(f"  Model is not wrapped → {type(base_model).__name__}")

    if explainer_type == "tree":
        explainer = shap.TreeExplainer(base_model)
        print(f"  TreeExplainer created ")

    elif explainer_type == "kernel":
        # KernelExplainer works on the ORIGINAL wrapped model's predict_proba
        # because we want calibrated probabilities, not raw ones
        n_bg      = min(50, X_test.shape[0])
        bg        = shap.sample(X_test, n_bg, random_state=42)
        explainer = shap.KernelExplainer(model.predict_proba, bg)
        print(f"  KernelExplainer created   (background: {n_bg} samples)")

    return explainer


print("make_explainer() updated ")

make_explainer() updated 


In [15]:
def get_shap_values(explainer, explainer_type, X_test, disease):
    """
    Computes SHAP values and returns a clean 2D array (class 1 = disease).

    For tree models: use up to 500 samples (speed)
    For kernel models: use up to 200 samples (kernel is slower)

    SHAP can return different shapes depending on model type:
    - list of arrays  → take index [1] (class 1)
    - 3D array        → take [:, :, 1]
    - 2D array        → use directly
    """
    if explainer_type == "tree":
        n = min(500, X_test.shape[0])
    else:
        n = min(200, X_test.shape[0])

    X_sample = X_test[:n]
    print(f"  Computing SHAP on {n} samples...")

    raw = explainer.shap_values(X_sample)

    # Normalize output shape → always 2D (n_samples, n_features)
    if isinstance(raw, list):
        vals = raw[1] if len(raw) > 1 else raw[0]
        print(f"  Output: list of {len(raw)} → using class 1")
    elif isinstance(raw, np.ndarray) and raw.ndim == 3:
        vals = raw[:, :, 1]
        print(f"  Output: 3D array → slicing class 1")
    else:
        vals = raw
        print(f"  Output: 2D array → using directly")

    print(f"  SHAP values shape: {vals.shape} ")
    return vals, X_sample


print("get_shap_values() defined ")

get_shap_values() defined 


In [22]:
def save_outputs(disease, model_key, features,
                 shap_vals, X_sample, explainer):
    display = DISEASE_CONFIG[disease]["display_name"]
    base    = f"../explainability/{disease}"
    X_df    = pd.DataFrame(X_sample, columns=features)

    # ── 1. Save raw SHAP values ──────────────────────────────────────
    joblib.dump(shap_vals, f"{base}_shap_values.pkl")
    print(f"  Saved: {disease}_shap_values.pkl")

    # ── 2. Feature importance JSON ───────────────────────────────────
    mean_abs   = np.abs(shap_vals).mean(axis=0)
    importance = {f: round(float(v), 6)
                  for f, v in zip(features, mean_abs)}
    importance = dict(sorted(importance.items(),
                              key=lambda x: x[1], reverse=True))

    ev = getattr(explainer, "expected_value", 0.0)
    if isinstance(ev, (list, np.ndarray)):
        ev = float(ev[1]) if len(ev) > 1 else float(ev[0])

    summary = {
        "disease"           : disease,
        "model"             : model_key,
        "base_value"        : float(ev),
        "feature_importance": importance,
        "top_5_features"    : list(importance.keys())[:5],
        "top_10_features"   : list(importance.keys())[:10],
        "n_features"        : len(features),
        "n_samples_used"    : int(shap_vals.shape[0]),
    }
    with open(f"{base}_shap.json", "w") as f:
        json.dump(summary, f, indent=4)
    print(f"  Saved: {disease}_shap.json  |  top 5: {summary['top_5_features']}")

    # ── 3. Bar chart ─────────────────────────────────────────────────
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_df,
                      plot_type="bar", max_display=15, show=False)
    plt.title(f"{display} — Global Feature Importance (SHAP)",
              fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{base}_shap_bar.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {disease}_shap_bar.png")

    # ── 4. Beeswarm ──────────────────────────────────────────────────
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_vals, X_df,
                      plot_type="dot", max_display=15, show=False)
    plt.title(f"{display} — Feature Impact (Beeswarm)",
              fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{base}_shap_beeswarm.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {disease}_shap_beeswarm.png")

    # ── 5. Waterfall ─────────────────────────────────────────────────
    shap_exp = shap.Explanation(
        values       = shap_vals[0],
        base_values  = float(ev),
        data         = X_df.iloc[0].values,
        feature_names= features,
    )
    plt.figure(figsize=(10, 8))
    shap.waterfall_plot(shap_exp, max_display=12, show=False)
    plt.title(f"{display} — Single Patient Explanation",
              fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{base}_shap_waterfall.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {disease}_shap_waterfall.png")

    # ── 6. Save explainer (tree) or background data (kernel) ─────────
    explainer_type = DISEASE_CONFIG[disease]["explainer"]
    if explainer_type == "tree":
        joblib.dump(explainer, f"{base}_explainer.pkl")
        print(f"  Saved: {disease}_explainer.pkl  (TreeExplainer)")
    else:
        # KernelExplainer cannot be pickled (Cython memory objects)
        # Save background data instead — dashboard recreates explainer
        joblib.dump(X_sample[:50], f"{base}_explainer_background.pkl")
        print(f"  Saved: {disease}_explainer_background.pkl  (kernel background)")

    print(f"\n  {display.upper()} — ALL OUTPUTS SAVED ")
    return summary


print("save_outputs() updated ")

save_outputs() updated 


---

## Step 3 — Master Runner

One function that calls all four above in sequence for one disease.
Run one disease at a time. Wait for completion before the next.

In [17]:
def run(disease):
    """
    Runs the full SHAP pipeline for one disease.
    Call this once per disease, one at a time.
    """
    cfg   = DISEASE_CONFIG[disease]
    print(f"\n{'='*55}")
    print(f"  {cfg['display_name'].upper()}")
    print(f"  Model: {cfg['model_key'].upper()}  |  "
          f"Explainer: {cfg['explainer'].capitalize()}")
    print(f"{'='*55}")

    print("\n[1] Loading data...")
    model, features, X_test, y_test = load_data(disease)

    print("\n[2] Creating explainer...")
    explainer = make_explainer(model, cfg["explainer"], X_test)

    print("\n[3] Computing SHAP values...")
    shap_vals, X_sample = get_shap_values(
        explainer, cfg["explainer"], X_test, disease
    )

    print("\n[4] Saving all outputs...")
    summary = save_outputs(
        disease, cfg["model_key"],
        features, shap_vals, X_sample, explainer
    )

    return {
        "shap_vals": shap_vals,
        "X_sample" : X_sample,
        "features" : features,
        "explainer": explainer,
        "summary"  : summary,
    }


print("run() defined ")
print("\nReady. Run each disease cell below one at a time.")

run() defined 

Ready. Run each disease cell below one at a time.


---

## Step 4 — Run Each Disease

Run one cell at a time. Wait for it to finish before the next.

Expected times:
| Disease | Explainer | Time |
|---|---|---|
| Diabetes | TreeExplainer (500 samples) | 2–4 min |
| Heart | KernelExplainer (200 samples) | 5–8 min |
| Liver | TreeExplainer (500 samples) | 3–5 min |
| Kidney | KernelExplainer (80 samples) | 1–2 min |

In [20]:
result_diabetes = run("diabetes")


  DIABETES
  Model: XGB  |  Explainer: Tree

[1] Loading data...
  Model    : CalibratedClassifierCV
  Features : 21
  X_test   : (14139, 21)

[2] Creating explainer...
  Unwrapped CalibratedClassifierCV → XGBClassifier
  TreeExplainer created 

[3] Computing SHAP values...
  Computing SHAP on 500 samples...
  Output: 2D array → using directly
  SHAP values shape: (500, 21) 

[4] Saving all outputs...
  Saved: diabetes_shap_values.pkl
  Saved: diabetes_shap.json  |  top 5: ['GenHlth', 'BMI', 'HighBP', 'Age', 'HighChol']
  Saved: diabetes_shap_bar.png
  Saved: diabetes_shap_beeswarm.png
  Saved: diabetes_shap_waterfall.png
  Saved: diabetes_explainer.pkl

  DIABETES — ALL OUTPUTS SAVED 


In [23]:
result_heart = run("heart")


  HEART DISEASE
  Model: LR  |  Explainer: Kernel

[1] Loading data...
  Model    : CalibratedClassifierCV
  Features : 25
  X_test   : (48494, 25)

[2] Creating explainer...
  Unwrapped CalibratedClassifierCV → LogisticRegression
  KernelExplainer created   (background: 50 samples)

[3] Computing SHAP values...
  Computing SHAP on 200 samples...


100%|██████████| 200/200 [00:58<00:00,  3.44it/s]


  Output: 3D array → slicing class 1
  SHAP values shape: (200, 25) 

[4] Saving all outputs...
  Saved: heart_shap_values.pkl
  Saved: heart_shap.json  |  top 5: ['AgeCategory', 'GeneralHealth', 'ChestScan', 'Sex', 'HadDiabetes']
  Saved: heart_shap_bar.png
  Saved: heart_shap_beeswarm.png
  Saved: heart_shap_waterfall.png
  Saved: heart_explainer_background.pkl  (kernel background)

  HEART DISEASE — ALL OUTPUTS SAVED 


In [24]:
result_liver = run("liver")


  LIVER DISEASE
  Model: RF  |  Explainer: Tree

[1] Loading data...
  Model    : RandomForestClassifier
  Features : 10
  X_test   : (6139, 10)

[2] Creating explainer...
  Model is not wrapped → RandomForestClassifier
  TreeExplainer created 

[3] Computing SHAP values...
  Computing SHAP on 500 samples...
  Output: 3D array → slicing class 1
  SHAP values shape: (500, 10) 

[4] Saving all outputs...
  Saved: liver_shap_values.pkl
  Saved: liver_shap.json  |  top 5: ['Direct Bilirubin', 'Total Bilirubin', 'Sgpt Alamine Aminotransferase', 'Alkphos Alkaline Phosphotase', 'Sgot Aspartate Aminotransferase']
  Saved: liver_shap_bar.png
  Saved: liver_shap_beeswarm.png
  Saved: liver_shap_waterfall.png
  Saved: liver_explainer.pkl  (TreeExplainer)

  LIVER DISEASE — ALL OUTPUTS SAVED 


In [25]:
result_kidney = run("kidney")


  CHRONIC KIDNEY DISEASE
  Model: LR  |  Explainer: Kernel

[1] Loading data...
  Model    : CalibratedClassifierCV
  Features : 24
  X_test   : (80, 24)

[2] Creating explainer...
  Unwrapped CalibratedClassifierCV → LogisticRegression
  KernelExplainer created   (background: 50 samples)

[3] Computing SHAP values...
  Computing SHAP on 80 samples...


100%|██████████| 80/80 [00:21<00:00,  3.65it/s]


  Output: 3D array → slicing class 1
  SHAP values shape: (80, 24) 

[4] Saving all outputs...
  Saved: kidney_shap_values.pkl
  Saved: kidney_shap.json  |  top 5: ['sg', 'pcv', 'dm', 'sc', 'appet']
  Saved: kidney_shap_bar.png
  Saved: kidney_shap_beeswarm.png
  Saved: kidney_shap_waterfall.png
  Saved: kidney_explainer_background.pkl  (kernel background)

  CHRONIC KIDNEY DISEASE — ALL OUTPUTS SAVED 


---

## Step 5 — Verify All Outputs

Run this after all four diseases are done.
Every row should show "OK".
If anything shows "MISSING" — re-run just that disease cell.

In [28]:
print("="*55)
print("VERIFICATION — ALL EXPLAINABILITY OUTPUTS")
print("="*55)

expected = []
for d in ["diabetes", "heart", "liver", "kidney"]:
    explainer_type = DISEASE_CONFIG[d]["explainer"]
    for suffix in [
        "_shap_values.pkl",
        "_shap.json",
        "_shap_bar.png",
        "_shap_beeswarm.png",
        "_shap_waterfall.png",
    ]:
        expected.append(f"../explainability/{d}{suffix}")

    # tree models save explainer.pkl, kernel models save background.pkl
    if explainer_type == "tree":
        expected.append(f"../explainability/{d}_explainer.pkl")
    else:
        expected.append(f"../explainability/{d}_explainer_background.pkl")

all_ok = True
for path in expected:
    exists = os.path.exists(path)
    status = " OK" if exists else " MISSING"
    if not exists:
        all_ok = False
    print(f"  {os.path.basename(path):<50} {status}")

print("\n" + "="*55)
if all_ok:
    print("ALL 24 FILES PRESENT ")
    print("Push explainability/ to GitHub")
else:
    print("SOME FILES MISSING — re-run the failing disease cell.")
print("="*55)

VERIFICATION — ALL EXPLAINABILITY OUTPUTS
  diabetes_shap_values.pkl                            OK
  diabetes_shap.json                                  OK
  diabetes_shap_bar.png                               OK
  diabetes_shap_beeswarm.png                          OK
  diabetes_shap_waterfall.png                         OK
  diabetes_explainer.pkl                              OK
  heart_shap_values.pkl                               OK
  heart_shap.json                                     OK
  heart_shap_bar.png                                  OK
  heart_shap_beeswarm.png                             OK
  heart_shap_waterfall.png                            OK
  heart_explainer_background.pkl                      OK
  liver_shap_values.pkl                               OK
  liver_shap.json                                     OK
  liver_shap_bar.png                                  OK
  liver_shap_beeswarm.png                             OK
  liver_shap_waterfall.png                    

---

## Step 6 — Cross-Disease Comparison Chart

One combined chart showing top features for all 4 diseases.
Used in the presentation and report.
Reads from the JSON files — no SHAP recomputation needed.

In [29]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

diseases = ["diabetes", "heart", "liver", "kidney"]
colors   = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"]

for i, (d, col) in enumerate(zip(diseases, colors)):
    with open(f"../explainability/{d}_shap.json") as f:
        data = json.load(f)

    imp   = data["feature_importance"]
    feats = list(imp.keys())[:10]
    vals  = [imp[f] for f in feats]

    axes[i].barh(range(len(feats)), vals,
                 color=col, alpha=0.85, edgecolor="white")
    axes[i].set_yticks(range(len(feats)))
    axes[i].set_yticklabels(feats, fontsize=9)
    axes[i].invert_yaxis()
    axes[i].set_xlabel("Mean |SHAP Value|", fontsize=10)
    axes[i].set_title(
        f"{DISEASE_CONFIG[d]['display_name']}\n"
        f"(Model: {DISEASE_CONFIG[d]['model_key'].upper()})",
        fontsize=12, fontweight="bold"
    )
    axes[i].grid(axis="x", alpha=0.3, linestyle="--")

plt.suptitle(
    "MediSight AI — Feature Importance Across All Diseases\nSHAP Analysis",
    fontsize=16, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig("../explainability/all_diseases_shap_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: all_diseases_shap_comparison.png ")

Saved: all_diseases_shap_comparison.png 
